# HyDE, Hypothetical Document Embeddings

A question and the passage that answers it often share almost no words. "What could go wrong?" is answered by "Risk factors include market volatility." A retriever that compares the question to the corpus has very little to hold on to.

HyDE fixes the query instead of the corpus. The model drafts a short passage that would answer the question, and that draft becomes the search query. The draft speaks the language of a 10-K filing, so it lands closer to the real passages. Nobody ever reads the draft and it does not have to be true. It is only a better-shaped query.

## Load in documents and chunk them

HyDE changes the query and never the corpus, so the chunks are exactly the ones the dense notebook uses.

In [1]:
from rag.documents import load_documents
from rag.chunk import chunk_documents

FILE_PATH = "/home/nick/github-projects/Sec-Rag/data/google_10K.pdf"

documents = load_documents(FILE_PATH)

chunks = chunk_documents(
    documents=documents,
    chunk_size=500,
    chunk_overlap=50
)

print(f"Length of Documents: {len(documents)}")
print(f"Length of Chunks: {len(chunks)}")

/home/nick/github-projects/Sec-Rag/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Length of Documents: 107
Length of Chunks: 810


## Build the base retriever

HyDE wraps another retriever rather than replacing it. Here the base is the dense retriever, so both the draft and the chunks are compared as embeddings.

In [2]:
from rag.dense import DenseRetriever

retriever = DenseRetriever()
retriever.add_documents(chunks)

print(f"Indexed {len(retriever)} chunks")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8641.65it/s]


Indexed 810 chunks


## Create the generator and wrap the retriever

The generator is a langchain chat model behind the small `Generator` interface the rest of `rag/` uses. `gpt-4o-mini` is the model here; `load_dotenv()` reads `OPENAI_API_KEY` from `.env`.

`HydeRetriever` takes the base retriever and the generator. Every call to `retrieve` drafts a passage first and then hands the draft to the base retriever.

In [3]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from rag.hyde import HydeRetriever
from rag.llm import DEFAULT_CHAT_MODEL, LangChainGenerator

load_dotenv()

generator = LangChainGenerator(ChatOpenAI(model=DEFAULT_CHAT_MODEL, max_tokens=512))
hyde = HydeRetriever(base_retriever=retriever, generator=generator)

## Look at the draft

This is the passage the retriever actually searches with. It is written in the tone of a filing and invents plausible numbers, which is fine because it is only used as a query.

In [4]:
from rag.prompts import HYDE_PROMPT

question = "what could go wrong for the company?"
draft = generator.generate(HYDE_PROMPT.format(question=question))

print(draft)

In evaluating the potential risks associated with our business operations, we acknowledge several key factors that may adversely impact our financial performance and market position. First, we are susceptible to fluctuating raw material costs, which can significantly affect margins; prices for critical components, particularly lithium used in our battery production, have seen volatility of up to 30% over the past year. Additionally, regulatory changes related to environmental standards could impose additional compliance costs, with estimates suggesting a potential increase of 15% in operational expenditures should proposed legislation come into effect. Furthermore, the ongoing chip shortage has hindered our manufacturing capacity; any further delays could exacerbate our supply chain disruptions and result in loss of market share as competitors capitalize on uninterrupted production. Lastly, cybersecurity threats pose an increasing risk; we are in the process of enhancing our data prote

## Compare plain dense retrieval with HyDE

For each question, the pages the base retriever returns are printed next to the pages HyDE returns, followed by the answer generated over the HyDE context.

In [5]:
from rag.pipeline import RAGPipeline


def pages(docs):
    return [d.metadata["page"] for d in docs]


pipeline = RAGPipeline(hyde, generator, top_k=4)

for i, q in enumerate([
    "how much did Google spend on research and development in 2025?",
    "how much did Google spend on research and development in 2024?",
    "what were Google's main operating costs?",
    "what are some pending acquisitions the company faces?",
]):
    dense_docs = retriever.retrieve(q, top_k=4)
    answer = pipeline.answer(q)
    print(f"\n━━━ {i + 1} Q: {q}")
    print(f"dense pages: {pages(dense_docs)}")
    print(f"hyde pages:  {pages(answer.documents)}")
    print(f"> {answer.text}")


━━━ 1 Q: how much did Google spend on research and development in 2025?
dense pages: [41, 40, 40, 44]
hyde pages:  [32, 3, 3, 41]
> Google spent $61,087 million on research and development in 2025. (Source: [/home/nick/github-projects/Sec-Rag/data/google_10K.pdf p41])

━━━ 2 Q: how much did Google spend on research and development in 2024?
dense pages: [41, 40, 40, 39]
hyde pages:  [3, 32, 40, 84]
> The context provided does not contain information regarding how much Google spent on research and development in 2024.

━━━ 3 Q: what were Google's main operating costs?
dense pages: [94, 42, 42, 5]
hyde pages:  [94, 38, 41, 36]
> Google's main operating costs included employee compensation expenses, costs related to legal matters, depreciation expenses, and content acquisition costs, particularly associated with YouTube. In total, the cost of revenues was $162.5 billion, with operating expenses amounting to $111.3 billion. Employee compensation expenses and other general expenses signific